# Notebook 28 — Corrected benchmarks and the structure–performance statistics

**Author:** Sunita Sahu · **Revision items:** A4, A2, A1, A10, B4

Every published benchmark score was checked against its primary source (`benchmarks_v2.csv` records the split, metric, prompting, subset and table for each). Structural predictors come from notebook 26 (attribution, trainable share) and notebook 27 (visual tokens, compute). All tests are exploratory.

---
## Setup

In [1]:
import sys, math, itertools
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
from vlm_audit.correlation_grid import (spearman_with_partial, spearman_permutation_test, benjamini_hochberg,
                                        benjamini_yekutieli, exact_two_group_test, has_variation)
from vlm_audit.analytic_compute import fisher_interval
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)

benchmarks = pd.read_csv(MEASUREMENT_DIR / 'benchmarks_v2.csv')
components = pd.read_csv(MEASUREMENT_DIR / 'nb26_parameter_components_v2.csv').set_index('model')
trainable = pd.read_csv(MEASUREMENT_DIR / 'nb26_trainable_share_v2.csv').set_index('model')
tokens = pd.read_csv(MEASUREMENT_DIR / 'nb27_table4_tokens_v2.csv').set_index('model')
compute_1024 = pd.read_csv(MEASUREMENT_DIR / 'nb27_table3_v2_1024.csv', index_col=0)
compute_448 = pd.read_csv(MEASUREMENT_DIR / 'nb27_table3_v2_448.csv', index_col=0)
RELEASE_YEAR = {'CLIP-ViT-L/14': 2021, 'SigLIP-L/16': 2023, 'SigLIP2-So400m/14': 2025, 'ViLT-B/32': 2021, 'BLIP-large': 2022,
                'BLIP-2-OPT-2.7B': 2023, 'InstructBLIP-Vicuna-7B': 2023, 'LLaVA-1.5-7B': 2023, 'LLaVA-NeXT-7B': 2024,
                'Molmo-7B-D': 2024, 'IDEFICS-9B': 2023, 'Llama-3.2-11B-Vision': 2024, 'Idefics2-8B': 2024, 'Qwen2-VL-7B': 2024,
                'Qwen2.5-VL-7B': 2025, 'InternVL3-8B': 2025, 'Pixtral-12B': 2024, 'DeepSeek-VL2-Small': 2024,
                'Chameleon-7B': 2024, 'Janus-Pro-7B': 2025, 'Qwen3-VL-8B': 2025, 'Gemma-4-12B': 2026, 'Gemma-4-26B-A4B': 2026}
BENCHMARKS = ['ai2d', 'chartqa', 'docvqa', 'mathvista', 'mmbench_en', 'mmmu', 'seedbench']
INSTRUCTION_TUNED = ['InstructBLIP-Vicuna-7B', 'LLaVA-1.5-7B', 'LLaVA-NeXT-7B', 'Idefics2-8B', 'Qwen2-VL-7B', 'Qwen2.5-VL-7B',
                     'InternVL3-8B', 'Pixtral-12B', 'Llama-3.2-11B-Vision', 'DeepSeek-VL2-Small', 'Janus-Pro-7B', 'Molmo-7B-D',
                     'Qwen3-VL-8B', 'Gemma-4-12B', 'Gemma-4-26B-A4B']

---
## 1. Table 6 rebuilt (A4)

In [2]:
status_counts = benchmarks['status'].str.split(' ').str[0].value_counts()
print('Status of the published and added cells:', status_counts.to_dict())
changes = benchmarks[~benchmarks['status'].str.startswith('confirmed')][['model', 'benchmark', 'published_value', 'score', 'status', 'note']]
changes

Status of the published and added cells: {'confirmed': 36, 'corrected': 20, 'not': 16, 'added': 14, 'removed': 6}


,model,benchmark,published_value,score,status,note
0,InstructBLIP-Vicuna-7B,seedbench,53.4,58.8,corrected (subset),benchmark-authors' evaluation; published 53.4 ...
1,InstructBLIP-Vicuna-7B,mmbench_en,NaN,38.3,added,benchmark-authors' evaluation
2,InstructBLIP-Vicuna-7B,vqav2,75.2,NaN,removed,no primary source; VQAv2 is a held-in training...
4,LLaVA-1.5-7B,seedbench,58.6,66.1,corrected (subset),published 58.6 was the image+video overall score
6,LLaVA-NeXT-7B,ai2d,66.6,60.8,corrected (variant),published value was the Vicuna-7B variant; tra...
7,LLaVA-NeXT-7B,chartqa,54.8,38.8,corrected (variant),published value was the Vicuna-7B variant
8,LLaVA-NeXT-7B,docvqa,74.4,72.2,corrected (variant),published value was the Vicuna-7B variant; val...
9,LLaVA-NeXT-7B,mathvista,NaN,37.7,added,NaN
10,LLaVA-NeXT-7B,mmbench_en,67.4,68.7,corrected (variant),published value was the Vicuna-7B variant
11,LLaVA-NeXT-7B,mmmu,35.1,35.3,corrected (variant),NaN


In [3]:
valid = benchmarks.dropna(subset=['score'])
table6_v2 = valid.pivot(index='model', columns='benchmark', values='score').reindex(columns=BENCHMARKS + ['vqav2'])
table6_v2.to_csv(MEASUREMENT_DIR / 'nb28_table6_v2.csv')
coverage = valid[valid['model'].isin(INSTRUCTION_TUNED) & valid['benchmark'].isin(BENCHMARKS)].groupby('benchmark')['model'].count()
print('Instruction-tuned models per benchmark (published: AI2D 7, ChartQA 8, DocVQA 9, MathVista 8, MMBench 8, MMMU 10, SEED 5):')
print(coverage.to_dict())
table6_v2

Instruction-tuned models per benchmark (published: AI2D 7, ChartQA 8, DocVQA 9, MathVista 8, MMBench 8, MMMU 10, SEED 5):
{'ai2d': 8, 'chartqa': 9, 'docvqa': 10, 'mathvista': 10, 'mmbench_en': 9, 'mmmu': 11, 'seedbench': 4}


benchmark,ai2d,chartqa,docvqa,mathvista,mmbench_en,mmmu,seedbench,vqav2
model,,,,,,,,
BLIP-2-OPT-2.7B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.5
DeepSeek-VL2-Small,80.0,84.5,92.3,60.7,82.3,48.0,NaN,NaN
IDEFICS-9B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.9
Idefics2-8B,NaN,NaN,74.0,51.4,76.7,43.0,NaN,81.2
InstructBLIP-Vicuna-7B,NaN,NaN,NaN,NaN,38.3,NaN,58.8,NaN
InternVL3-8B,85.2,86.6,92.7,71.6,83.4,62.7,NaN,NaN
Janus-Pro-7B,NaN,NaN,NaN,NaN,79.2,41.0,72.1,NaN
LLaVA-1.5-7B,NaN,NaN,NaN,NaN,64.3,NaN,66.1,78.5
LLaVA-NeXT-7B,60.8,38.8,72.2,37.7,68.7,35.3,72.2,82.2


In [4]:
# Protocol heterogeneity that remains after correction (validity threat for every correlation).
in_grid = valid[valid['model'].isin(INSTRUCTION_TUNED) & valid['benchmark'].isin(BENCHMARKS)]
heterogeneity = in_grid.groupby('benchmark').agg(
    n=('model', 'count'),
    splits=('split', lambda values: ', '.join(sorted(set(values.fillna('n/s'))))),
    chain_of_thought=('prompting', lambda values: int((values == 'chain-of-thought').sum())),
    subsets_or_versions=('subset_or_version', lambda values: ', '.join(sorted(set(values.fillna('-'))))),
    non_first_party=('source', lambda values: int(values.str.contains('SEED-Bench paper|MMBench paper|blog 2024-05-10').sum())))
heterogeneity.to_csv(MEASUREMENT_DIR / 'nb28_protocol_heterogeneity.csv')
heterogeneity

,n,splits,chain_of_thought,subsets_or_versions,non_first_party
benchmark,,,,,
ai2d,8,test,0,"-, with mask, with mask (opaque boxes), with m...",1
chartqa,9,test,2,-,1
docvqa,10,"test, val",0,"-, 320 tokens (image splitting)",1
mathvista,10,"testmini, testmini (VLMEvalKit convention)",1,"-, 320 tokens (image splitting)",0
mmbench_en,9,"dev, dev (repository convention), not stated, ...",0,"320 tokens (image splitting), not stated, v1.0",1
mmmu,11,"not stated, val, val (convention), val (from c...",2,"-, 320 tokens (image splitting), MMMU (standar...",0
seedbench,4,"SEED-Bench v1, not stated",0,"image subset, subset not stated",1


---
## 2. Structural predictors

Seven predictors (A1 adds V%, log V/L and F% to the published four). Token counts use the common-input convention of notebook 27: tokens at the 1024 × 1024 reference image and the released maximum.

In [5]:
predictors = pd.DataFrame({
    'trainable_pct': trainable['revised_pct'],
    'log tokens_1024': np.log(tokens['tokens_1024']),
    'log tokens_max': np.log(tokens['tokens_max']),
    'log total_params': np.log(components['total_params']),
    'V_pct': components['V_pct'],
    'log V_over_L': np.log(components['V_over_L']),
    'F_pct': components['F_pct'],
})
predictors['year'] = pd.Series(RELEASE_YEAR)
PREDICTORS = ['trainable_pct', 'log tokens_1024', 'log tokens_max', 'log total_params', 'V_pct', 'log V_over_L', 'F_pct']
predictors.loc[INSTRUCTION_TUNED].round(3)

,trainable_pct,log tokens_1024,log tokens_max,log total_params,V_pct,log V_over_L,F_pct,year
model,,,,,,,,
InstructBLIP-Vicuna-7B,2.39,3.466,3.466,22.792,12.459,-1.902,2.386,2023
LLaVA-1.5-7B,95.70,6.356,6.356,22.678,4.297,-3.081,0.297,2023
LLaVA-NeXT-7B,100.00,7.982,7.982,22.747,4.011,-3.154,0.277,2024
Idefics2-8B,100.00,5.787,5.787,22.852,4.972,-2.834,8.845,2024
Qwen2-VL-7B,92.39,7.223,9.704,22.838,7.613,-2.416,0.538,2024
Qwen2.5-VL-7B,92.38,7.223,9.704,22.839,7.621,-2.415,0.538,2025
InternVL3-8B,100.00,7.849,8.111,22.796,3.827,-3.146,0.347,2025
Pixtral-12B,100.00,8.333,8.333,23.264,3.181,-3.357,0.248,2024
Llama-3.2-11B-Vision,24.74,8.765,8.765,23.091,8.093,-2.371,0.295,2024


---
## 3. The extended grid (A1), effective group size (A2), BH and BY (A10)

Permutation p-values: every ordering up to n = 10, 20,000 Monte Carlo orderings above. A model whose trainable share is unknown (Qwen3-VL-8B, source silent) is left out of the trainable-share tests only. Families: the 49 marginal tests and the 49 year-partial tests, each adjusted separately. `n_low_trainable` counts models in the grid sample whose trainable share is below 50% (frozen language model).

In [6]:
def benchmark_sample(benchmark, models=INSTRUCTION_TUNED):
    """Models with a score on one benchmark, with their scores."""
    rows = valid[(valid['benchmark'] == benchmark) & valid['model'].isin(models)]
    return rows.set_index('model')['score']

def run_grid(models=INSTRUCTION_TUNED):
    """All benchmark x predictor tests for one model set."""
    rows = []
    for benchmark in BENCHMARKS:
        scores = benchmark_sample(benchmark, models)
        for predictor in PREDICTORS:
            known = predictors.loc[scores.index, predictor].notna()   # pairwise complete
            sample = scores[known.values]
            structural = predictors.loc[sample.index, predictor]
            row = {'benchmark': benchmark, 'predictor': predictor, 'n': len(sample),
                   'n_low_trainable': int((predictors.loc[sample.index, 'trainable_pct'] < 50).sum())}
            if len(sample) >= 4 and has_variation(structural) and has_variation(sample):
                result = spearman_with_partial(structural.values, sample.values, predictors.loc[sample.index, 'year'].values)
                interval = fisher_interval(result['rho'], result['sample_size']) if result['sample_size'] >= 8 else None
                row.update({'rho': result['rho'], 'p_perm': result['p_value'], 'ci_low': interval[0] if interval else np.nan,
                            'ci_high': interval[1] if interval else np.nan, 'partial_rho_year': result['partial_rho'],
                            'partial_p': result['partial_p_value']})
            rows.append(row)
    grid = pd.DataFrame(rows)
    tested = grid['p_perm'].notna()
    for column, adjusted in (('p_perm', 'p_bh'), ('partial_p', 'partial_p_bh')):
        grid.loc[tested, adjusted] = benjamini_hochberg(grid.loc[tested, column])
    grid.loc[tested, 'p_by'] = benjamini_yekutieli(grid.loc[tested, 'p_perm'])
    grid.loc[tested, 'partial_p_by'] = benjamini_yekutieli(grid.loc[tested, 'partial_p'])
    return grid

grid = run_grid()
grid.to_csv(MEASUREMENT_DIR / 'nb28_table8_v2.csv', index=False)
print(f"{int(grid['p_perm'].notna().sum())} marginal tests; minimum adjusted p: BH {grid['p_bh'].min():.3f}, BY {grid['p_by'].min():.3f}; "
      f"year-partial BH {grid['partial_p_bh'].min():.3f}, BY {grid['partial_p_by'].min():.3f}")
print('Unadjusted p < 0.05:', int((grid['p_perm'] < 0.05).sum()), 'marginal,', int((grid['partial_p'] < 0.05).sum()), 'partial')
grid.round(3)

49 marginal tests; minimum adjusted p: BH 0.176, BY 0.790; year-partial BH 0.328, BY 1.000
Unadjusted p < 0.05: 4 marginal, 2 partial


,benchmark,predictor,n,n_low_trainable,rho,p_perm,ci_low,ci_high,partial_rho_year,partial_p,p_bh,partial_p_bh,p_by,partial_p_by
0,ai2d,trainable_pct,7,1,-0.670,0.114,NaN,NaN,-0.717,0.129,0.566,0.528,1.00,1.0
1,ai2d,log tokens_1024,8,1,0.084,0.849,-0.660,0.745,0.237,0.635,0.946,0.950,1.00,1.0
2,ai2d,log tokens_max,8,1,0.586,0.139,-0.203,0.913,0.437,0.353,0.566,0.787,1.00,1.0
3,ai2d,log total_params,8,1,0.357,0.389,-0.464,0.848,0.448,0.332,0.831,0.776,1.00,1.0
4,ai2d,V_pct,8,1,0.619,0.115,-0.152,0.922,0.628,0.128,0.566,0.528,1.00,1.0
5,ai2d,log V_over_L,8,1,0.738,0.046,0.070,0.949,0.720,0.059,0.561,0.510,1.00,1.0
6,ai2d,F_pct,8,1,0.381,0.360,-0.443,0.856,0.228,0.651,0.831,0.950,1.00,1.0
7,chartqa,trainable_pct,8,1,-0.191,0.673,-0.789,0.594,-0.126,0.810,0.946,0.979,1.00,1.0
8,chartqa,log tokens_1024,9,1,-0.544,0.135,-0.887,0.188,-0.519,0.191,0.566,0.586,1.00,1.0
9,chartqa,log tokens_max,9,1,0.288,0.455,-0.465,0.799,-0.185,0.653,0.885,0.950,1.00,1.0


In [7]:
# Sensitivity: drop scores that are not first-party (benchmark-authors' or re-evaluation numbers).
first_party = valid[~valid['source'].str.contains('SEED-Bench paper|MMBench paper|blog 2024-05-10')]
original_valid = valid
valid = first_party
grid_first_party = run_grid()
valid = original_valid
print(f"First-party scores only: minimum BH {grid_first_party['p_bh'].min():.3f} (marginal), "
      f"{grid_first_party['partial_p_bh'].min():.3f} (partial)")
grid_first_party.to_csv(MEASUREMENT_DIR / 'nb28_table8_v2_first_party_only.csv', index=False)

First-party scores only: minimum BH 0.283 (marginal), 0.281 (partial)


### A2 — the trainable-share rows as a two-group comparison

With the revised shares, the low-trainable group in each benchmark sample is the frozen-language-model design (InstructBLIP on MMBench and SEED-Bench, Llama-3.2-Vision on five benchmarks). The exact Mann–Whitney test below enumerates every assignment of the group label; with one member its smallest attainable p-value is 2/n.

In [8]:
two_group_rows = []
for benchmark in BENCHMARKS:
    scores = benchmark_sample(benchmark)
    scores = scores[predictors.loc[scores.index, 'trainable_pct'].notna().values]   # unknown share: left out
    low = predictors.loc[scores.index, 'trainable_pct'] < 50
    result = exact_two_group_test(scores.values, low.values)
    marginal = grid[(grid['benchmark'] == benchmark) & (grid['predictor'] == 'trainable_pct')].iloc[0]
    two_group_rows.append({'benchmark': benchmark, 'n': result['n'], 'n_low_trainable': result['n1'],
                           'low_trainable_models': ', '.join(scores.index[low]),
                           'mean_rank_difference': result['mean_rank_difference'], 'p_exact': result['p_value'],
                           'minimum_attainable_p': result['minimum_attainable_p'], 'spearman_rho': marginal.get('rho')})
two_group = pd.DataFrame(two_group_rows).set_index('benchmark')
two_group.to_csv(MEASUREMENT_DIR / 'nb28_trainable_two_group.csv')
two_group.round(3)

,n,n_low_trainable,low_trainable_models,mean_rank_difference,p_exact,minimum_attainable_p,spearman_rho
benchmark,,,,,,,
ai2d,7,1,Llama-3.2-11B-Vision,3.500,0.286,0.286,-0.670
chartqa,8,1,Llama-3.2-11B-Vision,-0.571,1.000,0.250,-0.191
docvqa,9,1,Llama-3.2-11B-Vision,-2.250,0.667,0.222,-0.347
mathvista,9,1,Llama-3.2-11B-Vision,-2.250,0.667,0.222,-0.050
mmbench_en,9,1,InstructBLIP-Vicuna-7B,-4.500,0.222,0.222,0.096
mmmu,10,1,Llama-3.2-11B-Vision,0.556,1.000,0.200,-0.294
seedbench,4,1,InstructBLIP-Vicuna-7B,-2.000,0.500,0.500,1.000


---
## 4. Lineage-pair sensitivity (A10)

Six lineage groups: five pairs and the Qwen-VL family (Qwen2-VL, Qwen2.5-VL, Qwen3-VL); the Gemma 4 pair has no benchmark scores. For every test, each subset that keeps exactly one member of every group with two or more members in the sample is re-tested.

In [9]:
LINEAGE_GROUPS = [('LLaVA-1.5-7B', 'LLaVA-NeXT-7B'), ('Qwen2-VL-7B', 'Qwen2.5-VL-7B', 'Qwen3-VL-8B'),
                  ('BLIP-2-OPT-2.7B', 'InstructBLIP-Vicuna-7B'), ('IDEFICS-9B', 'Idefics2-8B'),
                  ('SigLIP-L/16', 'SigLIP2-So400m/14'), ('Gemma-4-12B', 'Gemma-4-26B-A4B')]

def lineage_subsets(names):
    """Every subset of names keeping one member of each lineage group with two or more members present."""
    present = [[member for member in group if member in names] for group in LINEAGE_GROUPS]
    present = [members for members in present if len(members) >= 2]
    for choice in itertools.product(*present):
        dropped = {member for members, kept in zip(present, choice) for member in members if member != kept}
        yield [name for name in names if name not in dropped]

sensitivity_rows = []
for benchmark in BENCHMARKS:
    scores = benchmark_sample(benchmark)
    for predictor in PREDICTORS:
        values = []
        known = scores[predictors.loc[scores.index, predictor].notna().values]   # pairwise complete
        for subset in lineage_subsets(list(known.index)):
            structural = predictors.loc[subset, predictor]
            if len(subset) >= 4 and has_variation(structural) and has_variation(scores[subset]):
                test = spearman_permutation_test(structural.values, scores[subset].values)
                values.append((test['rho'], test['p_value']))
        if values:
            rhos, p_values = zip(*values)
            sensitivity_rows.append({'benchmark': benchmark, 'predictor': predictor, 'subsets': len(values),
                                     'rho_min': min(rhos), 'rho_max': max(rhos), 'p_min': min(p_values),
                                     'sign_changes': int(min(rhos) < 0 < max(rhos))})
lineage = pd.DataFrame(sensitivity_rows)
lineage.to_csv(MEASUREMENT_DIR / 'nb28_lineage_sensitivity.csv', index=False)
print('Tests whose sign depends on which lineage member is kept:', int(lineage['sign_changes'].sum()), 'of', len(lineage))
lineage.round(3)

Tests whose sign depends on which lineage member is kept: 8 of 42


,benchmark,predictor,subsets,rho_min,rho_max,p_min,sign_changes
0,ai2d,trainable_pct,2,-0.676,-0.676,0.200,0
1,ai2d,log tokens_1024,3,0.143,0.257,0.658,0
2,ai2d,log tokens_max,3,0.600,0.714,0.136,0
3,ai2d,log total_params,3,0.257,0.371,0.497,0
4,ai2d,V_pct,3,0.543,0.657,0.175,0
5,ai2d,log V_over_L,3,0.771,0.829,0.058,0
6,ai2d,F_pct,3,0.371,0.543,0.297,0
7,chartqa,trainable_pct,2,-0.223,0.134,0.667,1
8,chartqa,log tokens_1024,3,-0.607,-0.357,0.167,0
9,chartqa,log tokens_max,3,-0.321,0.214,0.498,1


In [10]:
# rho(V%, C_V%) over the 19 separable models: all 2^5 = 32 one-per-pair subsets (compute at 1024 x 1024).
separable = [name for name in components.index if name != 'Chameleon-7B']
rho_rows = []
for subset in lineage_subsets(separable):
    test = spearman_permutation_test(components.loc[subset, 'V_pct'].values,
                                     compute_1024.loc[subset, 'vision_compute_share_pct'].values)
    rho_rows.append({'n': len(subset), 'rho': test['rho'], 'p': test['p_value']})
rho_lineage = pd.DataFrame(rho_rows)
print(f"{len(rho_lineage)} subsets of n = {rho_lineage['n'].iloc[0]}: rho from {rho_lineage['rho'].min():+.3f} to "
      f"{rho_lineage['rho'].max():+.3f}; largest p {rho_lineage['p'].max():.4f}")
rho_lineage.to_csv(MEASUREMENT_DIR / 'nb28_vision_share_correlation_lineage.csv', index=False)

96 subsets of n = 15: rho from +0.629 to +0.850; largest p 0.0140


---
## 5. Numbers quoted in Sections 5.4–5.7, recomputed

In [11]:
year_rows = []
for benchmark in BENCHMARKS:
    scores = benchmark_sample(benchmark)
    test = spearman_permutation_test(predictors.loc[scores.index, 'year'].values, scores.values)
    year_rows.append({'benchmark': benchmark, 'n': len(scores), 'rho_year': test['rho'], 'p_perm': test['p_value']})
year_correlation = pd.DataFrame(year_rows).set_index('benchmark')
year_correlation.to_csv(MEASUREMENT_DIR / 'nb28_year_correlation.csv')
print('Section 5.7: release year vs score (published: MMB +0.82, SEED +0.79, ChartQA +0.76, MathVista +0.76, DocVQA +0.62, AI2D +0.47, MMMU +0.34)')
year_correlation.round(3)

Section 5.7: release year vs score (published: MMB +0.82, SEED +0.79, ChartQA +0.76, MathVista +0.76, DocVQA +0.62, AI2D +0.47, MMMU +0.34)


,n,rho_year,p_perm
benchmark,,,
ai2d,8,0.507,0.250
chartqa,9,0.822,0.024
docvqa,10,0.722,0.033
mathvista,10,0.798,0.017
mmbench_en,9,0.811,0.013
mmmu,11,0.478,0.163
seedbench,4,0.738,0.333


In [12]:
# Section 5.4: does the token-accounting convention flip the sign of the bandwidth-score correlation?
convention_rows = []
for benchmark in BENCHMARKS:
    scores = benchmark_sample(benchmark)
    row = {'benchmark': benchmark, 'n': len(scores)}
    for column in ('tokens_448', 'tokens_1024', 'tokens_max'):
        row[f'rho_{column}'] = spearman_permutation_test(np.log(tokens.loc[scores.index, column]).values, scores.values)['rho']
    convention_rows.append(row)
conventions = pd.DataFrame(convention_rows).set_index('benchmark')
conventions['sign_flips'] = conventions.apply(lambda row: int(min(row.iloc[1:]) < 0 < max(row.iloc[1:])), axis=1)
conventions.to_csv(MEASUREMENT_DIR / 'nb28_bandwidth_conventions.csv')
conventions.round(3)

,n,rho_tokens_448,rho_tokens_1024,rho_tokens_max,sign_flips
benchmark,,,,,
ai2d,8,-0.464,0.084,0.586,1
chartqa,9,-0.712,-0.544,0.288,1
docvqa,10,-0.828,-0.304,0.669,1
mathvista,10,-0.706,-0.085,0.509,1
mmbench_en,9,-0.170,0.519,0.837,1
mmmu,11,-0.706,0.187,0.780,1
seedbench,4,1.000,1.000,1.000,0


In [13]:
# Section 5.5: frozen-language-model model vs the trainable median, per benchmark.
median_rows = []
for benchmark in BENCHMARKS:
    scores = benchmark_sample(benchmark)
    scores = scores[predictors.loc[scores.index, 'trainable_pct'].notna().values]   # unknown share: left out
    low = predictors.loc[scores.index, 'trainable_pct'] < 50
    for name in scores.index[low]:
        others = scores[~low]
        median_rows.append({'benchmark': benchmark, 'frozen_LLM_model': name, 'score': scores[name],
                            'trainable_median': others.median(), 'n_trainable': len(others),
                            'difference': scores[name] - others.median()})
frozen_vs_trainable = pd.DataFrame(median_rows)
frozen_vs_trainable.to_csv(MEASUREMENT_DIR / 'nb28_frozen_vs_trainable_medians.csv', index=False)
frozen_vs_trainable.round(2)

,benchmark,frozen_LLM_model,score,trainable_median,n_trainable,difference
0,ai2d,Llama-3.2-11B-Vision,91.1,82.70,6,8.40
1,chartqa,Llama-3.2-11B-Vision,83.4,84.10,7,-0.70
2,docvqa,Llama-3.2-11B-Vision,88.4,92.25,8,-3.85
3,mathvista,Llama-3.2-11B-Vision,51.5,58.25,8,-6.75
4,mmbench_en,InstructBLIP-Vicuna-7B,38.3,80.75,8,-42.45
5,mmmu,Llama-3.2-11B-Vision,50.7,48.00,9,2.70
6,seedbench,InstructBLIP-Vicuna-7B,58.8,72.10,3,-13.30


In [14]:
# Section 5.6: Idefics2 against Qwen2-VL on the benchmarks both report.
shared = [benchmark for benchmark in BENCHMARKS if pd.notna(table6_v2.loc['Idefics2-8B', benchmark]) and pd.notna(table6_v2.loc['Qwen2-VL-7B', benchmark])]
gaps = (table6_v2.loc['Qwen2-VL-7B', shared] - table6_v2.loc['Idefics2-8B', shared]).round(1)
print('Shared benchmarks:', shared)
print('Qwen2-VL minus Idefics2:', gaps.to_dict(), f'-> range {gaps.min()} to {gaps.max()} points (published: 5.3 to 20.5 on five)')
print('Fusion-boundary tokens at 1024 x 1024: Idefics2', int(tokens.loc['Idefics2-8B', 'tokens_1024']), '| Qwen2-VL', int(tokens.loc['Qwen2-VL-7B', 'tokens_1024']))

Shared benchmarks: ['docvqa', 'mathvista', 'mmbench_en', 'mmmu']
Qwen2-VL minus Idefics2: {'docvqa': 20.5, 'mathvista': 6.8, 'mmbench_en': 6.3, 'mmmu': 11.1} -> range 6.3 to 20.5 points (published: 5.3 to 20.5 on five)
Fusion-boundary tokens at 1024 x 1024: Idefics2 326 | Qwen2-VL 1371


In [15]:
# Section 3.4 / 6.3: within-lineage benchmark deltas (published: LLaVA-NeXT - LLaVA-1.5 = 3.4, Qwen2.5-VL - Qwen2-VL = 3.5).
pair_rows = []
for newer, older in (('LLaVA-NeXT-7B', 'LLaVA-1.5-7B'), ('Qwen2.5-VL-7B', 'Qwen2-VL-7B')):
    deltas = (table6_v2.loc[newer, BENCHMARKS] - table6_v2.loc[older, BENCHMARKS]).dropna()
    pair_rows.append({'pair': f'{newer} - {older}', 'n_shared': len(deltas), 'mean_delta': deltas.mean(),
                      'deltas': ', '.join(f'{name} {value:+.1f}' for name, value in deltas.items())})
matched_pairs = pd.DataFrame(pair_rows).set_index('pair')
matched_pairs.to_csv(MEASUREMENT_DIR / 'nb28_matched_pair_deltas.csv')
matched_pairs.round(2)

,n_shared,mean_delta,deltas
pair,,,
LLaVA-NeXT-7B - LLaVA-1.5-7B,2,5.25,"mmbench_en +4.4, seedbench +6.1"
Qwen2.5-VL-7B - Qwen2-VL-7B,6,3.57,"ai2d +0.9, chartqa +4.3, docvqa +1.2, mathvist..."


---
## 6. Section 5.8 efficiency on corrected scores and revised compute (B4)

In [16]:
def pareto_flags(scores, compute):
    """True where no other model has score >= and compute <= with at least one strict."""
    flags = []
    for index, (score, cost) in enumerate(zip(scores, compute)):
        dominated = any(other_score >= score and other_cost <= cost and (other_score > score or other_cost < cost)
                        for other_index, (other_score, other_cost) in enumerate(zip(scores, compute)) if other_index != index)
        flags.append(not dominated)
    return flags

def efficiency(compute_table, benchmark=None):
    """Composite (or one-benchmark) score, TFLOPs, score per TFLOP and Pareto flag."""
    rows = []
    for name in INSTRUCTION_TUNED:
        available = table6_v2.loc[name, BENCHMARKS].dropna() if name in table6_v2.index else pd.Series(dtype=float)
        if benchmark is not None:
            available = available[available.index == benchmark]
        if available.empty:
            continue
        teraflops = compute_table.loc[name, 'total_g'] / 1000
        rows.append({'model': name, 'n_benchmarks': len(available), 'score': available.mean(), 'TFLOPs': teraflops,
                     'score_per_TFLOP': available.mean() / teraflops})
    table = pd.DataFrame(rows)
    table['pareto'] = pareto_flags(table['score'], table['TFLOPs'])
    return table.sort_values('TFLOPs').set_index('model')

pareto_summary = {}
for input_label, compute_table in (('1024', compute_1024), ('448', compute_448)):
    for label, benchmark in (('composite', None), ('docvqa', 'docvqa'), ('mmmu', 'mmmu')):
        table = efficiency(compute_table, benchmark)
        table.to_csv(MEASUREMENT_DIR / f'nb28_efficiency_{label}_{input_label}.csv')
        pareto_summary[(input_label, label)] = sorted(table.index[table['pareto']])
for key, members in pareto_summary.items():
    print(f'{key[1]:9s} at {key[0]:>4s}px input: Pareto set {members}')
print()
print('Published Pareto sets: composite {InstructBLIP, Idefics2, DeepSeek-VL2, InternVL3}; MMMU {Idefics2, DeepSeek-VL2, Llama-3.2, InternVL3}')
efficiency(compute_1024).round(2)

composite at 1024px input: Pareto set ['DeepSeek-VL2-Small', 'InstructBLIP-Vicuna-7B', 'Janus-Pro-7B', 'LLaVA-1.5-7B', 'Qwen3-VL-8B']
docvqa    at 1024px input: Pareto set ['DeepSeek-VL2-Small', 'Qwen3-VL-8B']
mmmu      at 1024px input: Pareto set ['DeepSeek-VL2-Small', 'Janus-Pro-7B', 'Llama-3.2-11B-Vision', 'Qwen3-VL-8B']
composite at  448px input: Pareto set ['InstructBLIP-Vicuna-7B', 'Llama-3.2-11B-Vision', 'Qwen3-VL-8B']
docvqa    at  448px input: Pareto set ['Llama-3.2-11B-Vision', 'Qwen3-VL-8B']
mmmu      at  448px input: Pareto set ['Llama-3.2-11B-Vision', 'Qwen3-VL-8B']

Published Pareto sets: composite {InstructBLIP, Idefics2, DeepSeek-VL2, InternVL3}; MMMU {Idefics2, DeepSeek-VL2, Llama-3.2, InternVL3}


,n_benchmarks,score,TFLOPs,score_per_TFLOP,pareto
model,,,,,
InstructBLIP-Vicuna-7B,2,48.55,1.37,35.42,True
Janus-Pro-7B,3,64.10,8.00,8.02,True
LLaVA-1.5-7B,2,65.20,8.47,7.69,True
DeepSeek-VL2-Small,6,74.63,17.07,4.37,True
Molmo-7B-D,5,71.12,17.53,4.06,False
Idefics2-8B,4,61.28,20.06,3.05,False
Llama-3.2-11B-Vision,5,73.02,20.33,3.59,False
Qwen3-VL-8B,5,83.64,21.16,3.95,True
Qwen2.5-VL-7B,6,79.53,26.80,2.97,False


In [17]:
composite = efficiency(compute_1024)
print(f"Composite score range {composite['score'].min():.1f}-{composite['score'].max():.1f} ({composite['score'].max()/composite['score'].min():.2f}x); "
      f"compute {composite['TFLOPs'].min():.2f}-{composite['TFLOPs'].max():.2f} TFLOPs; "
      f"score/TFLOP {composite['score_per_TFLOP'].min():.2f}-{composite['score_per_TFLOP'].max():.2f} "
      f"({composite['score_per_TFLOP'].max()/composite['score_per_TFLOP'].min():.1f}x)")
pixtral_vs_qwen = compute_1024.loc['Pixtral-12B', 'total_g'] / compute_1024.loc['Qwen2.5-VL-7B', 'total_g']
print(f'Pixtral / Qwen2.5-VL compute at 1024 x 1024: {pixtral_vs_qwen:.1f}x (published: about 20x); at 448 x 448: '
      f"{compute_448.loc['Pixtral-12B', 'total_g'] / compute_448.loc['Qwen2.5-VL-7B', 'total_g']:.1f}x")

Composite score range 48.5-83.6 (1.72x); compute 1.37-108.16 TFLOPs; score/TFLOP 0.65-35.42 (54.2x)
Pixtral / Qwen2.5-VL compute at 1024 x 1024: 4.0x (published: about 20x); at 448 x 448: 3.8x
